In [5]:
import math
import random
from collections import Counter

# Les mêmes fonctions que pour C4.5(sont deja expliquer)
def calculer_entropie(donnees, index_cible):
    valeurs_cibles = [ligne[index_cible] for ligne in donnees]
    compteur = Counter(valeurs_cibles)
    total = len(donnees)
    return sum(- (nb / total) * math.log2(nb / total) for nb in compteur.values())

def separer_donnees(donnees, index_attribut, valeur):
    return [ligne for ligne in donnees if ligne[index_attribut] == valeur]

def information_division(donnees, index_attribut):
    valeurs = [ligne[index_attribut] for ligne in donnees]
    compteur = Counter(valeurs)
    total = len(donnees)
    return sum(- (nb / total) * math.log2(nb / total) for nb in compteur.values())

def ratio_gain(donnees, index_attribut, index_cible):
    entropie_avant = calculer_entropie(donnees, index_cible)
    valeurs_uniques = set([ligne[index_attribut] for ligne in donnees])
    entropie_apres = 0.0
    total = len(donnees)
    
    for valeur in valeurs_uniques:
        sous_ensemble = separer_donnees(donnees, index_attribut, valeur)
        entropie_apres += (len(sous_ensemble) / total) * calculer_entropie(sous_ensemble, index_cible)
        
    gain_info = entropie_avant - entropie_apres
    split_info = information_division(donnees, index_attribut)
    
    return 0.0 if split_info == 0 else gain_info / split_info

In [2]:
def construire_arbre_aleatoire(donnees, noms_attributs, index_cible):
    valeurs_cibles = [ligne[index_cible] for ligne in donnees]
    
    # Conditions d'arrêt
    if len(set(valeurs_cibles)) == 1:
        return valeurs_cibles[0]
    if len(noms_attributs) == 0:
        return Counter(valeurs_cibles).most_common(1)[0][0]
    
    # On ne sélectionne qu'un sous-ensemble d'attributs
    nb_attributs_total = len(noms_attributs)
    # Mathématiquement, on prend souvent la racine carrée
    nb_a_choisir = max(1, int(math.sqrt(nb_attributs_total))) 
    
    # Choix aléatoire des indices des attributs à tester
    indices_aleatoires = random.sample(range(nb_attributs_total), nb_a_choisir)
    
    # On calcule le gain uniquement pour ces attributs tirés au sort
    ratios = []
    for i in indices_aleatoires:
        ratio = ratio_gain(donnees, i, index_cible)
        ratios.append((ratio, i)) # On garde le ratio ET l'index
        
    # On prend le meilleur parmi ceux sélectionnés
    meilleur_ratio, index_meilleur = max(ratios, key=lambda x: x[0])
    meilleur_attribut = noms_attributs[index_meilleur]
    
    arbre = {meilleur_attribut: {}}
    valeurs_possibles = set([ligne[index_meilleur] for ligne in donnees])
    
    attributs_restants = noms_attributs.copy()
    attributs_restants.pop(index_meilleur)
    
    for valeur in valeurs_possibles:
        sous_ensemble = separer_donnees(donnees, index_meilleur, valeur)
        
        sous_ensemble_nettoye = []
        for ligne in sous_ensemble:
            nouvelle_ligne = ligne[:index_meilleur] + ligne[index_meilleur+1:]
            sous_ensemble_nettoye.append(nouvelle_ligne)
            
        if len(sous_ensemble_nettoye) == 0:
            arbre[meilleur_attribut][valeur] = Counter(valeurs_cibles).most_common(1)[0][0]
        else:
            nouvel_index_cible = index_cible - 1 if index_cible > index_meilleur else index_cible
            arbre[meilleur_attribut][valeur] = construire_arbre_aleatoire(sous_ensemble_nettoye, attributs_restants, nouvel_index_cible)
        
    return arbre

In [3]:
def entrainer_foret_aleatoire(donnees, noms_attributs, index_cible, nb_arbres=5):
    foret = []
    taille_donnees = len(donnees)
    
    for i in range(nb_arbres):
        # BOOTSTRAPPING : Créer un échantillon aléatoire de même taille, avec remise
        echantillon_bootstrap = [random.choice(donnees) for _ in range(taille_donnees)]
        
        # Entraîner un arbre sur cet échantillon spécifique
        arbre = construire_arbre_aleatoire(echantillon_bootstrap, noms_attributs, index_cible)
        foret.append(arbre)
        
    return foret

# Fonction de prédiction d'un SEUL arbre 
def prediction_un_arbre(arbre, ligne_donnee, noms_attributs):
    noeud_actuel = list(arbre.keys())[0]
    index_attribut = noms_attributs.index(noeud_actuel)
    valeur_donnee = ligne_donnee[index_attribut]
    
    try:
        sous_arbre = arbre[noeud_actuel][valeur_donnee]
    except KeyError:
        return None # Retourne None si la valeur est inconnue pour cet arbre
    
    if isinstance(sous_arbre, dict):
        return prediction_un_arbre(sous_arbre, ligne_donnee, noms_attributs)
    else:
        return sous_arbre

# Prédiction de la FORÊT
def prediction_foret(foret, ligne_donnee, noms_attributs):
    votes = []
    for arbre in foret:
        vote = prediction_un_arbre(arbre, ligne_donnee, noms_attributs)
        if vote is not None:
            votes.append(vote)
            
    if len(votes) == 0:
        return "Indéterminé (Valeurs inconnues par tous les arbres)"
        
    # Retourner le vote majoritaire
    compteur_votes = Counter(votes)
    prediction_finale = compteur_votes.most_common(1)[0][0]
    
    return prediction_finale, compteur_votes

In [6]:
# Notre jeu de données météo classique
noms_colonnes = ["Meteo", "Temperature", "Humidite", "Vent"]

donnees_entrainement = [
    ["Soleil", "Chaud", "Haute", "Faible", "Non"],
    ["Soleil", "Chaud", "Haute", "Fort", "Non"],
    ["Nuages", "Chaud", "Haute", "Faible", "Oui"],
    ["Pluie", "Doux", "Haute", "Faible", "Oui"],
    ["Pluie", "Frais", "Normale", "Faible", "Oui"],
    ["Pluie", "Frais", "Normale", "Fort", "Non"],
    ["Nuages", "Frais", "Normale", "Fort", "Oui"],
    ["Soleil", "Doux", "Haute", "Faible", "Non"],
    ["Soleil", "Frais", "Normale", "Faible", "Oui"],
    ["Pluie", "Doux", "Normale", "Faible", "Oui"],
    ["Soleil", "Doux", "Normale", "Fort", "Oui"],
    ["Nuages", "Doux", "Haute", "Fort", "Oui"],
    ["Nuages", "Chaud", "Normale", "Faible", "Oui"],
    ["Pluie", "Doux", "Haute", "Fort", "Non"]
]

# Nous créons 10 arbres
ma_foret = entrainer_foret_aleatoire(donnees_entrainement, noms_colonnes, index_cible=-1, nb_arbres=10)
print("Forêt plantée avec succès !\n")

# Faisons une prédiction
jour_test = ["Soleil", "Frais", "Normale", "Fort"]

prediction_finale, details_votes = prediction_foret(ma_foret, jour_test, noms_colonnes)

print(f"Condition météo à tester : {jour_test}")
print(f"Détails des votes des arbres : {dict(details_votes)}")
print(f"Verdict de la Forêt (Prédiction finale) : Doit-on jouer au tennis ? => {prediction_finale}")

Forêt plantée avec succès !

Condition météo à tester : ['Soleil', 'Frais', 'Normale', 'Fort']
------------------------------
Détails des votes des arbres : {'Oui': 9}
Verdict de la Forêt (Prédiction finale) : Doit-on jouer au tennis ? => Oui
